In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_final = pd.read_csv("/content/drive/MyDrive/data_handling_and_machine_learning/datasets/merged_dataset/merged_dataset.csv")

# 1. Inspections and verfications

## Inspect data types

In [ ]:
print(df_final.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 386 entries, 0 to 385
Data columns (total 45 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   SimFinId                                         386 non-null    int64  
 1   Ticker                                           386 non-null    object 
 2   Fiscal Year                                      386 non-null    int64  
 3   Publish Date                                     386 non-null    object 
 4   t0                                               386 non-null    object 
 5   Price at t0                                      386 non-null    float64
 6   One Year Date                                    386 non-null    object 
 7   t1                                               386 non-null    object 
 8   Price at t1                                      386 non-null    float64
 9   Future Return                   

### ✅ PASS

## Inspect target values

In [ ]:
df_final["Target"].value_counts().sort_index()

,count
Target,
0,144
1,242


### ✅ PASS

## Inspect missing target

In [ ]:
print("Unique target values:", sorted(df_final["Target"].unique()))
print("Missing target value:", df_final["Target"].isna().sum())

Unique target values: [np.int64(0), np.int64(1)]
Missing target value: 0


### ✅ PASS

## Inspect infinite numeric values

In [ ]:
df_numeric = df_final.select_dtypes(include="number")

infinite_counts = np.isinf(df_numeric).sum()

print(infinite_counts[infinite_counts > 0])

Series([], dtype: int64)


### ✅ PASS

## Verify target construction

In [ ]:
check_target = ((df_final["Future Return"] > 0).astype(int) == df_final["Target"])

print("Rows matching target rule:", check_target.sum())
print("Rows violating target rule:", (~check_target).sum())

Rows matching target rule: 386
Rows violating target rule: 0


### ✅ PASS

In [ ]:
print(df_final.columns.tolist())

['SimFinId', 'Ticker', 'Fiscal Year', 'Publish Date', 't0', 'Price at t0', 'One Year Date', 't1', 'Price at t1', 'Future Return', 'Target', 'Historical Return', 'Historical Volatility', 'Average Trading Volume', 'Revenue', 'Gross Profit', 'Operating Income', 'Net Income', 'Revenue Growth', 'Gross Profit Margin', 'Operating Profit Margin', 'Net Profit Margin', 'Operating Income Growth', 'Net Income Growth', 'Operating Cash Flow', 'Capital Expenditure', 'Free Cash Flow', 'Operating Cash Flow Growth', 'Free Cash Flow Growth', 'FCF Margin', 'Cash, Cash Equivalents & Short Term Investments', 'Total Current Assets', 'Total Assets', 'Total Current Liabilities', 'Total Liabilities', 'Total Equity', 'balance_diff', 'balance_error_pct', 'Short Term Debt', 'Long Term Debt', 'Current Ratio', 'Debt-to-Equity', 'Asset Growth', 'Equity Growth', 'Debt Growth']


## Date conversion

In [ ]:
date_columns =["Publish Date", "t0", "One Year Date", "t1"]

for col in date_columns:
  df_final[col] = pd.to_datetime(df_final[col])

df_final[date_columns].dtypes

,0
Publish Date,datetime64[ns]
t0,datetime64[ns]
One Year Date,datetime64[ns]
t1,datetime64[ns]


In [ ]:
print(df_final[["Publish Date", "t0", "One Year Date", "t1"]].isna().sum())

Publish Date     0
t0               0
One Year Date    0
t1               0
dtype: int64


### ✅ PASS

## Exclusion of unnecessary columns

```text
SimFinId
Ticker
Fiscal Year
Publish Date
t0
Price at t0
One Year Date
t1
Price at t1
Future Return
Target
Free Cash Flow
balance_diff
balance_error_pct
```

In [ ]:
excluded_columns = ["SimFinId", "Ticker", "Fiscal Year", "Publish Date", "t0", "Price at t0", "One Year Date", "t1", "Price at t1", "Future Return", "Target", "Free Cash Flow", "balance_diff", "balance_error_pct"]

candidate_features = [col for col in df_final.columns
                      if col not in excluded_columns
                      ]

print("Candidate feature count:", len(candidate_features))
print(candidate_features)

Candidate feature count: 31
['Historical Return', 'Historical Volatility', 'Average Trading Volume', 'Revenue', 'Gross Profit', 'Operating Income', 'Net Income', 'Revenue Growth', 'Gross Profit Margin', 'Operating Profit Margin', 'Net Profit Margin', 'Operating Income Growth', 'Net Income Growth', 'Operating Cash Flow', 'Capital Expenditure', 'Operating Cash Flow Growth', 'Free Cash Flow Growth', 'FCF Margin', 'Cash, Cash Equivalents & Short Term Investments', 'Total Current Assets', 'Total Assets', 'Total Current Liabilities', 'Total Liabilities', 'Total Equity', 'Short Term Debt', 'Long Term Debt', 'Current Ratio', 'Debt-to-Equity', 'Asset Growth', 'Equity Growth', 'Debt Growth']


### ✅ PASS

In [ ]:
correlation_matrix = df_final[candidate_features].corr()

high_correlations = (
    correlation_matrix
    .where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)

print(high_correlations.head(20))

Operating Income                                 Net Income                                         0.942670
Total Assets                                     Total Liabilities                                  0.932510
Operating Income                                 Operating Cash Flow                                0.930352
Net Income                                       Operating Cash Flow                                0.893677
Total Current Assets                             Total Current Liabilities                          0.893070
Gross Profit                                     Operating Cash Flow                                0.887058
Operating Cash Flow                              Total Equity                                       0.873567
Total Assets                                     Total Equity                                       0.867434
Revenue                                          Gross Profit                                       0.847282
Asset Growth       

## Checkpoint

In [ ]:
missing = df_final[candidate_features].isna().sum()

missing[missing > 0].sort_values(ascending=False)

,0
Free Cash Flow Growth,109
Short Term Debt,107
Equity Growth,97
Revenue Growth,97
Operating Income Growth,97
Asset Growth,97
Operating Cash Flow Growth,97
Net Income Growth,97
Debt Growth,97
Historical Return,96


# 2. Define the Feature Matrix and Target

Before training a machine learning model, we need to separate the dataset into:

- **Features (`X`)** — the information available at the prediction point that the model can use to make a prediction.
- **Target (`y`)** — the outcome that we want the model to predict.

Our target is `Target`, where:

- `1` = positive 1-year future stock return
- `0` = non-positive 1-year future stock return

The `candidate_features` list contains the 31 features selected for the first ML model.

Columns related to future information, identifiers, dates, and data-quality checks are not included in `X`.

In [ ]:
X = df_final[candidate_features]
y = df_final["Target"]

print("X shape:", X.shape)
print("Y shape:", y.shape)

X shape: (386, 31)
Y shape: (386,)


The feature matrix contains 386 observations and 31 candidate features. The target contains one binary label for each observation.

The number of rows in `X` and `y` matches, so each observation has a corresponding target value.

## Validate the Selected ML Features

Before splitting the dataset into training and testing sets, we perform a final check of the selected features.

The feature matrix should contain only information that is available at the prediction point. Identifiers, dates, future information, and accounting validation columns should not be used as model inputs.

We also verify the number of selected features and check for duplicate feature names.

In [ ]:
print("Number of features:", X.shape[1])

print("\nSelected features:")
for feature in X.columns:
  print("-", feature)

print("\nDuplicate feature names:", X.columns.duplicated().sum())

Number of features: 31

Selected features:
- Historical Return
- Historical Volatility
- Average Trading Volume
- Revenue
- Gross Profit
- Operating Income
- Net Income
- Revenue Growth
- Gross Profit Margin
- Operating Profit Margin
- Net Profit Margin
- Operating Income Growth
- Net Income Growth
- Operating Cash Flow
- Capital Expenditure
- Operating Cash Flow Growth
- Free Cash Flow Growth
- FCF Margin
- Cash, Cash Equivalents & Short Term Investments
- Total Current Assets
- Total Assets
- Total Current Liabilities
- Total Liabilities
- Total Equity
- Short Term Debt
- Long Term Debt
- Current Ratio
- Debt-to-Equity
- Asset Growth
- Equity Growth
- Debt Growth

Duplicate feature names: 0


# 3. Split the Data into Training and Testing Sets

Before handling missing values or training a model, we split the dataset into a training set and a testing set.

The training set is used to learn the model, while the testing set is kept separate and is used later to evaluate how well the model performs on unseen observations.

We use an 80/20 split:

- **80% training data**
- **20% testing data**

Because the target classes are not perfectly balanced, we use stratification. This keeps approximately the same proportion of positive and non-positive returns in both sets.

A fixed `random_state` is used so that the same split can be reproduced.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.20,
                                                    stratify=y,
                                                    random_state=42
                                                    )

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts(normalize=True))

print("\ny_test distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (308, 31)
X_test shape: (78, 31)

y_train distribution:
Target
1    0.626623
0    0.373377
Name: proportion, dtype: float64

y_test distribution:
Target
1    0.628205
0    0.371795
Name: proportion, dtype: float64


# 4. Handle Missing Values

The selected features contain missing values (`NaN`).

Instead of removing rows or replacing missing financial values with zero, we use median imputation.

For each feature, the median is calculated using the training data only. The learned median values are then used to fill missing values in both the training and testing sets.

This prevents information from the test set from influencing the preprocessing step and helps avoid data leakage.

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
imputer = SimpleImputer(strategy="median")

## Fit the Imputer on the Training Data

The imputer learns the median value of each feature from the training set.

We fit the imputer only on `X_train` so that information from the test set does not influence the preprocessing.

In [ ]:
imputer.fit(X_train)

SimpleImputer(strategy='median')

In [ ]:
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("X_train_imputed shape:", X_train_imputed.shape)
print("X_test_imputed shape:", X_test_imputed.shape)

X_train_imputed shape: (308, 31)
X_test_imputed shape: (78, 31)


## Verify the Imputed Data

After transformation, we verify that there are no remaining missing values in either the training or testing feature sets.

This confirms that the imputer successfully replaced the missing values using the medians learned from the training data.

In [ ]:
print("Missing values in X_train_imputed:", np.isnan(X_train_imputed).sum())
print("Missing values in X_test_imputed:", np.isnan(X_test_imputed).sum())

Missing values in X_train_imputed: 0
Missing values in X_test_imputed: 0


# 5. Restore Feature Names

The `SimpleImputer` returns NumPy arrays, so the column names from the original DataFrame are no longer attached.

We convert the imputed arrays back into pandas DataFrames and use the original feature names from `X.columns`.

This makes the processed data easier to inspect and allows us to identify features later when analyzing model coefficients or feature importance.

In [ ]:
X_train_imputed = pd.DataFrame(X_train_imputed, columns=X.columns, index=X_train.index)

X_test_imputed = pd.DataFrame(X_test_imputed, columns=X.columns, index=X_test.index)

In [ ]:
print("X_train_imputed shape:", X_train_imputed.shape)
print("X_test_imputed shape:", X_test_imputed.shape)

print("\nFirst 5 features:")
print(X_train_imputed.columns[:5].tolist())

X_train_imputed shape: (308, 31)
X_test_imputed shape: (78, 31)

First 5 features:
['Historical Return', 'Historical Volatility', 'Average Trading Volume', 'Revenue', 'Gross Profit']


# 6. Feature Scaling

The selected features have very different numerical scales. For example, financial amounts such as revenue can be measured in millions or billions, while ratios and growth rates are much smaller.

Standardization is useful for Logistic Regression because it puts the features on a comparable scale.

A Decision Tree does not require feature scaling because its decisions are based on thresholds rather than distances or feature magnitudes.

Therefore, we will use scaled features for Logistic Regression and the imputed but unscaled features for the Decision Tree.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [ ]:
scaler.fit(X_train_imputed)

StandardScaler()

In [ ]:
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [ ]:
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (308, 31)
X_test_scaled shape: (78, 31)


In [ ]:
print(X_train_scaled[:5, :5])

[[-0.58841385 -0.37391716  0.50828129  0.62308952  2.21304649]
 [-0.73912358  0.08071811 -0.10607058 -0.21164264  0.27602473]
 [-0.10410935 -0.20380767 -0.1997802  -0.64644314 -0.31460131]
 [ 1.12278432 -1.07090368 -0.21079961 -0.00898101  0.2990957 ]
 [ 0.35777671  0.66980154 -0.22707979  1.1194331  -0.14673423]]


# 7. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
# Create Logistic Regression Model
logistic_model = LogisticRegression(max_iter=1000)

In [ ]:
# Train Logistic Regression Model
logistic_model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
# Make predictions
y_pred_logistic = logistic_model.predict(X_test_scaled)

In [ ]:
print(y_pred_logistic[:20])

[1 1 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1]


## 7.1 Evaluate the Logistic Regression Model

After training the Logistic Regression model, we evaluate its performance on the testing set.

The evaluation is performed using several classification metrics:

- Accuracy
- Confusion Matrix
- Precision
- Recall
- F1-score

These metrics provide a more complete assessment than accuracy alone, especially when the target classes are not perfectly balanced.

In [ ]:
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score)

### 7.1.1 Accuracy

Accuracy measures the proportion of predictions that the model classified correctly.

It is calculated as:

Accuracy = (Number of Correct Predictions) / (Total Number of Predictions)

Although accuracy is a useful overall performance measure, it should not be interpreted alone because the target classes are not perfectly balanced. Therefore, additional metrics such as precision, recall, and the confusion matrix will also be evaluated.

In [ ]:
accuracy = accuracy_score(y_test, y_pred_logistic)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5641


**Interpretation:** The Logistic Regression model achieved an accuracy of 56.41%, meaning it correctly classified approximately 44 out of the 78 test observations.

### 7.1.2 Confusion Matrix

The confusion matrix summarizes the model's predictions by comparing the predicted classes with the true classes.

It reports:

- True Positives (TP)
- True Negatives (TN)
- False Positives (FP)
- False Negatives (FN)

This provides more detailed insight into the model's performance than accuracy alone.

In [ ]:
cm = confusion_matrix(y_test, y_pred_logistic)

print(cm)

[[ 5 24]
 [10 39]]


**Interpretation:** The model correctly identified 39 positive-return companies (TP) and 5 non-positive-return companies (TN). It incorrectly predicted positive returns for 24 companies (FP) and missed 10 companies that actually had positive returns (FN).

The relatively high number of false positives shows that the model tends to predict the positive class frequently.

### 7.1.3 Precision

Precision measures how many of the companies predicted as having a positive 1-year return actually had a positive return.

A higher precision means that positive predictions are more reliable.

In [ ]:
precision = precision_score(y_test, y_pred_logistic)

print(f"Precision: {precision:.4f}")

Precision: 0.6190


**Interpretation:** The model achieved a precision of 61.90%. This means that when the model predicted a positive 1-year return, it was correct approximately 62% of the time.

### 7.1.4 Recall

Recall measures how many of the companies that actually had a positive
1-year future return were correctly identified by the model.

A higher recall means the model misses fewer positive-return companies.

In [ ]:
recall = recall_score(y_test, y_pred_logistic)

print(f"Recall: {recall:.4f}")

Recall: 0.7959


**Interpretation:** The model correctly identified approximately 79.6% of
the companies that actually had a positive 1-year return.

### 7.1.5 F1-score

The F1-score combines precision and recall into a single metric.

It is useful when we want to consider both:
- how reliable the positive predictions are (precision), and
- how many actual positive cases the model identifies (recall).

The F1-score ranges from 0 to 1, where a higher value indicates better
performance.

In [ ]:
f1 = f1_score(y_test, y_pred_logistic)

print(f"F1 Score: {f1:.4f}")

F1 Score: 0.6964


**Interpretation:** The model achieved an F1-score of 69.64%, indicating a moderate balance between precision and recall.

### 7.1.6 Overall Interpretation

The Logistic Regression model achieved an accuracy of 56.41%, precision of 61.90%, recall of 79.59%, and F1-score of 69.64% on the test set.

The model has relatively high recall, meaning it identifies most of the companies with positive 1-year returns. However, its lower precision and the confusion matrix show that it also produces many false positive predictions.

These results provide a baseline for comparison with the Decision Tree model.

# 8. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# Create the Decision Tree model
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)

In [ ]:
# Train the Decision Tree
tree_model.fit(X_train_imputed, y_train)

DecisionTreeClassifier(max_depth=4, random_state=42)

In [ ]:
print("X_train_imputed exists:", "X_train_imputed" in globals())
print("y_train exists:", "y_train" in globals())

X_train_imputed exists: True
y_train exists: True


## 8.1 Making Predictions

After training the Decision Tree, we use it to predict the target values for the test set.

The model has not seen these observations during training, so the test set allows us to evaluate how well the trained model generalizes to unseen data.

In [ ]:
y_pred_tree = tree_model.predict(X_test_imputed)

print(y_pred_tree[:20])

[1 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 1 1 1 1]


### 8.1.1 Accuracy

Accuracy measures the proportion of test observations that the Decision Tree classified correctly.

We use the same metric as for Logistic Regression so that the two models can be compared fairly.

In [ ]:
accuracy_tree = accuracy_score(y_test, y_pred_tree)

print(f"Accuracy: {accuracy_tree:.4f}")

Accuracy: 0.6410


### 8.1.2 Confusion Matrix

The confusion matrix shows how the Decision Tree's predictions compare with the actual target values.

It allows us to see how many observations were correctly classified as positive or non-positive returns, as well as how many false positive and false negative predictions were made.

In [ ]:
cm_tree = confusion_matrix(y_test, y_pred_tree)

print(cm_tree)

[[ 5 24]
 [ 4 45]]


### 8.1.3 Precision

Precision measures how many of the companies predicted as having a positive 1-year return actually had a positive return.

A higher precision means that the model's positive predictions are more reliable.

In [ ]:
precision_tree = precision_score(y_test, y_pred_tree)

print(f"Precision of Decision Tree: {precision_tree:.4f}")

Precision of Decision Tree: 0.6522


### 8.1.4 Recall

Recall measures how many of the companies that actually had a positive 1-year return were correctly identified by the Decision Tree.

A higher recall means that the model misses fewer positive-return companies.

In [ ]:
recall_tree = recall_score(y_test, y_pred_tree)

print(f"Recall of Decision Tree: {recall_tree:.4f}")

Recall of Decision Tree: 0.9184


### 8.1.5 F1-score

The F1-score combines precision and recall into a single metric.

It is useful for evaluating the balance between the reliability of positive predictions and the ability to identify actual positive-return companies.

In [ ]:
f1_tree = f1_score(y_test, y_pred_tree)

print(f"F1-Score of Decision Tree: {f1_tree:.4f}")

F1-Score of Decision Tree: 0.7627


### 8.1.6 Overall Interpretation

The Decision Tree achieved an accuracy of 64.10%, precision of 65.22%, recall of 91.84%, and an F1-score of 76.27% on the test set.

The model achieved a high recall, correctly identifying most of the companies that had a positive 1-year return. However, it also produced 24 false positive predictions, which reduced its precision.

Compared with Logistic Regression, the Decision Tree achieved higher accuracy, precision, recall, and F1-score on the same test set.

These results suggest that the Decision Tree is currently the stronger of the two models for this dataset. However, the results are based on one train-test split, so they should be interpreted as a baseline comparison rather than proof that the Decision Tree will always perform better.

### Save the Final Classical ML Model

The `Decision Tree` was selected as the **final classical machine learning baseline** after comparing it with the majority-class baseline and Logistic Regression.

The model configuration is:

- Algorithm: Decision Tree Classifier
- `max_depth`: 4
- `random_state`: 42

The model is saved as a `.pkl` file so that the Streamlit application can load the already-trained model without retraining it.

In [ ]:
import joblib
import os

MODEL_PATH = ("/content/drive/MyDrive/data_handling_and_machine_learning/datasets/classical_ml_classification_dataset/InvestIQ_decision_tree_model.pkl")

joblib.dump(tree_model, MODEL_PATH)

print("Decision Tree model saved.")
print("Path:", MODEL_PATH)
print("Model:", tree_model)

Decision Tree model saved.
Path: /content/drive/MyDrive/data_handling_and_machine_learning/datasets/classical_ml_classification_dataset/InvestIQ_decision_tree_model.pkl
Model: DecisionTreeClassifier(max_depth=4, random_state=42)


### Save the Classical ML Feature List

The final classical ML model uses 31 selected features from the merged dataset.

The feature list is saved separately as a JSON file so that the Streamlit application can use the exact same feature names and feature order when preparing input data for prediction.

This prevents the application from accidentally using a different feature set or feature ordering than the one used during model training.

In [ ]:
import json

FEATURES_PATH = ("/content/drive/MyDrive/data_handling_and_machine_learning/datasets/classical_ml_classification_dataset/InvestIQ_classical_ml_features.json")

classical_ml_features = [
    "Historical Return",
    "Historical Volatility",
    "Average Trading Volume",
    "Revenue",
    "Gross Profit",
    "Operating Income",
    "Net Income",
    "Revenue Growth",
    "Gross Profit Margin",
    "Operating Profit Margin",
    "Net Profit Margin",
    "Operating Income Growth",
    "Net Income Growth",
    "Operating Cash Flow",
    "Capital Expenditure",
    "Operating Cash Flow Growth",
    "Free Cash Flow Growth",
    "FCF Margin",
    "Cash, Cash Equivalents & Short Term Investments",
    "Total Current Assets",
    "Total Assets",
    "Total Current Liabilities",
    "Total Liabilities",
    "Total Equity",
    "Short Term Debt",
    "Long Term Debt",
    "Current Ratio",
    "Debt-to-Equity",
    "Asset Growth",
    "Equity Growth",
    "Debt Growth"
]

with open(FEATURES_PATH, "w", encoding="utf-8") as f:
    json.dump(classical_ml_features, f, indent=2)

print("Feature list saved.")
print("Number of features:", len(classical_ml_features))
print("Path:", FEATURES_PATH)

Feature list saved.
Number of features: 31
Path: /content/drive/MyDrive/data_handling_and_machine_learning/datasets/classical_ml_classification_dataset/InvestIQ_classical_ml_features.json


### Save the Classical ML Imputer

The Decision Tree was trained using median-imputed features.

The imputer must be fitted only on the training data and then reused for new prediction data. Saving it separately ensures that the Streamlit application uses the same median values that were learned during model training.

This prevents information from the prediction data from influencing the preprocessing step.

In [ ]:
import joblib
from sklearn.impute import SimpleImputer

IMPUTER_PATH = ("/content/drive/MyDrive/data_handling_and_machine_learning/datasets/classical_ml_classification_dataset/InvestIQ_classical_ml_imputer.pkl")

# Recreate the same median imputation used during training
classical_ml_imputer = SimpleImputer(strategy="median")

classical_ml_imputer.fit(X_train)

joblib.dump(classical_ml_imputer, IMPUTER_PATH)

print("Imputer saved.")
print("Path:", IMPUTER_PATH)
print("Strategy:", classical_ml_imputer.strategy)

Imputer saved.
Path: /content/drive/MyDrive/data_handling_and_machine_learning/datasets/classical_ml_classification_dataset/InvestIQ_classical_ml_imputer.pkl
Strategy: median


### Verify the Saved Classical ML Artifacts

Load the saved Decision Tree, feature list, and imputer, then reproduce predictions on the existing test set.

The purpose of this check is to confirm that the saved artifacts behave consistently with the original trained model before integrating them into the Streamlit application.

In [ ]:
import json
import joblib
import pandas as pd

MODEL_PATH = ("/content/drive/MyDrive/data_handling_and_machine_learning/datasets/classical_ml_classification_dataset/InvestIQ_decision_tree_model.pkl")

FEATURES_PATH = ("/content/drive/MyDrive/data_handling_and_machine_learning/datasets/classical_ml_classification_dataset/InvestIQ_classical_ml_features.json")

IMPUTER_PATH = ("/content/drive/MyDrive/data_handling_and_machine_learning/datasets/classical_ml_classification_dataset/InvestIQ_classical_ml_imputer.pkl")

saved_model = joblib.load(MODEL_PATH)
saved_imputer = joblib.load(IMPUTER_PATH)

with open(FEATURES_PATH, "r", encoding="utf-8") as f:
    saved_features = json.load(f)

_test_saved = X_test[saved_features]

X_test_saved_imputed = saved_imputer.transform(X_test_saved)

# Restore feature names after imputation
X_test_saved_imputed = pd.DataFrame(
    X_test_saved_imputed,
    columns=saved_features,
    index=X_test_saved.index
)

saved_predictions = saved_model.predict(X_test_saved_imputed)

print("Model loaded:", type(saved_model).__name__)
print("Features loaded:", len(saved_features))
print("Imputer strategy:", saved_imputer.strategy)
print("Prediction count:", len(saved_predictions))
print("Warning-free prediction: successful")

Model loaded: DecisionTreeClassifier
Features loaded: 31
Imputer strategy: median
Prediction count: 78
Warning-free prediction: successful


### Verify Saved Model Against the Original Model

Compare predictions from the saved Decision Tree with predictions from the original trained `tree_model`.

The predictions should be identical because the saved model is the same trained model and the saved imputer reproduces the preprocessing used during training.

In [ ]:
from sklearn.metrics import accuracy_score

# Original model predictions
original_predictions = tree_model.predict(X_test_imputed)

# Saved model predictions
saved_predictions = saved_model.predict(X_test_saved_imputed)

predictions_identical = (
    original_predictions == saved_predictions
).all()

original_accuracy = accuracy_score(
    y_test,
    original_predictions
)

saved_accuracy = accuracy_score(
    y_test,
    saved_predictions
)

print("Predictions identical:", predictions_identical)
print(f"Original model accuracy: {original_accuracy:.4f}")
print(f"Saved model accuracy:   {saved_accuracy:.4f}")

Predictions identical: True
Original model accuracy: 0.6410
Saved model accuracy:   0.6410


# 9. Model Comparison

We trained two classification models using the same training and test data:

- Logistic Regression
- Decision Tree

Both models predict whether the company's 1-year future stock return is positive (`1`) or non-positive (`0`).

We compare their accuracy, precision, recall, and F1-score to evaluate their performance.

In [ ]:
comparison = pd.DataFrame({"Metric": ["Accuracy", "Precision", "Recall", "F1.score"],
                           "Logistic Regression": [accuracy, precision, recall, f1],
                           "Decision Tree": [accuracy_tree, precision_tree, recall_tree, f1_tree]
                           })

comparison

,Metric,Logistic Regression,Decision Tree
0,Accuracy,0.564103,0.641026
1,Precision,0.619048,0.652174
2,Recall,0.795918,0.918367
3,F1.score,0.696429,0.762712


# 10. Baseline Comparison

Before comparing machine learning models, it is useful to establish a simple baseline.

Our target contains two classes:

- `0` = non-positive 1-year return
- `1` = positive 1-year return

The majority class is `1`, which occurs in approximately 62.7% of the dataset.

A simple baseline model could therefore predict `1` for every test observation.

A trained machine learning model should ideally perform better than this simple baseline.

In [ ]:
baseline_predictions = np.ones(len(y_test), dtype=int)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"Baseline accuracy: {baseline_accuracy:.4f}")

Baseline accuracy: 0.6282


## 10.1 Baseline Metrics

Accuracy alone does not give the complete picture, so we also calculate precision, recall, and F1-score for the majority-class baseline.

This allows us to compare the baseline with both machine learning models using the same evaluation metrics.

In [ ]:
baseline_precision = precision_score(y_test, baseline_predictions)
baseline_recall = recall_score(y_test, baseline_predictions)
baseline_f1 = f1_score(y_test, baseline_predictions)

print(f"Baseline precision: {baseline_precision:.4f}")
print(f"Baseline recall: {baseline_recall:.4f}")
print(f"Baseline f1-score: {baseline_f1:.4f}")

Baseline precision: 0.6282
Baseline recall: 1.0000
Baseline f1-score: 0.7717


## 10.2 Baseline Confusion Matrix

The majority-class baseline predicts class `1` for every test observation.

Its confusion matrix shows that it identifies all positive-return companies, but it also incorrectly classifies every non-positive-return company as positive.

This demonstrates why accuracy and recall should not be interpreted without considering the types of errors made by the model.

In [ ]:
cm_baseline = confusion_matrix(y_test, baseline_predictions)

print(cm_baseline)

[[ 0 29]
 [ 0 49]]


# 11. Final Model Comparison

We now compare the majority-class baseline, Logistic Regression, and Decision Tree using the same test set and evaluation metrics.

The baseline provides a reference point for determining whether the machine learning models provide useful predictive performance beyond simply predicting the majority class.

In [ ]:
final_comparison = pd.DataFrame({"Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
                                 "Majority Baseline": [baseline_accuracy, baseline_precision, baseline_recall, baseline_f1],
                                 "Logistic Regression": [accuracy, precision, recall, f1],
                                 "Decision Tree": [accuracy_tree, precision_tree, recall_tree, f1_tree]
                                 })

final_comparison

,Metric,Majority Baseline,Logistic Regression,Decision Tree
0,Accuracy,0.628205,0.564103,0.641026
1,Precision,0.628205,0.619048,0.652174
2,Recall,1.000000,0.795918,0.918367
3,F1-score,0.771654,0.696429,0.762712


## 11.1 Final Interpretation

The Decision Tree achieved the highest accuracy (64.10%) and precision (65.22%) among the three approaches. It also achieved a high recall of 91.84%, meaning that it identified most of the companies that actually had a positive 1-year return.

Logistic Regression achieved lower performance than the majority-class baseline on all four evaluation metrics in this test split. Its accuracy was 56.41%, compared with 62.82% for the baseline.

The majority-class baseline achieved 100% recall and the highest F1-score (77.17%). However, this baseline simply predicts the positive class for every company. Its confusion matrix shows that it correctly identifies no non-positive-return companies and produces 29 false positive predictions.

The Decision Tree provides a more useful result because it performs slightly better than the majority-class baseline in accuracy and precision while also correctly identifying five non-positive-return companies. Therefore, among the models tested, the Decision Tree is the most promising baseline model for this dataset.

However, these results are based on a single train-test split of a relatively small dataset. They should therefore be treated as an initial baseline rather than evidence that the Decision Tree will always outperform other models.

# 12. Decision Tree Feature Importance

A Decision Tree assigns an importance score to each feature based on how much that feature contributes to the tree's decisions.

A higher importance score means that the feature was more useful for separating the target classes within the trained tree.

Feature importance does not mean that the feature causes the future stock return. It only describes how useful the feature was to this particular trained model.

In [ ]:
feature_importance = pd.Series(tree_model.feature_importances_,
                               index=candidate_features
                               ).sort_values(ascending=False)

feature_importance

,0
Total Equity,0.148604
Historical Return,0.131698
"Cash, Cash Equivalents & Short Term Investments",0.126161
Debt-to-Equity,0.124786
Capital Expenditure,0.121061
Operating Cash Flow Growth,0.119749
Total Assets,0.099632
Free Cash Flow Growth,0.079492
Debt Growth,0.048818
Net Income,0.000000


## 12.1 Interpretation

The Decision Tree assigned the highest feature-importance score to Total Equity (0.1486), followed by Historical Return (0.1317), Cash, Cash Equivalents & Short Term Investments (0.1262), and Debt-to-Equity (0.1248).

The tree used nine of the 31 selected features in its splits, while the remaining features received an importance score of zero. A zero score means that the feature was not used by this particular trained tree; it does not mean that the feature is generally unimportant.

Feature importance describes which features were useful for the decisions made by the trained model. It does not establish a causal relationship between a feature and future stock returns.

# 13. Final Dataset Sanity Check

Before moving on from the classical machine learning stage, we verify that the final dataset still has the expected structure and size.

This helps ensure that no rows or columns were accidentally added, removed, or duplicated during the feature engineering and modelling process.

In [ ]:
print("Dataset shape:", df_final.shape)
print("Number of unique companies:", df_final["SimFinId"].nunique())
print("Number of unique company-year combinations:", df_final[["SimFinId", "Fiscal Year"]].drop_duplicates().shape[0])

print("\nDuplicate company-year rows:", df_final.duplicated(subset=["SimFinId", "Fiscal Year"]).sum())

Dataset shape: (386, 45)
Number of unique companies: 97
Number of unique company-year combinations: 386

Duplicate company-year rows: 0


## 13.1 Target Sanity Check

The target variable represents whether the company's 1-year future stock return was positive.

We verify that the target contains only the expected binary values, has no missing observations, and matches the target-generation rule.

In [ ]:
print("Target values:")
print(df_final["Target"].value_counts().sort_index())

print("\nUnique target values:")
print(sorted(df_final["Target"].unique()))

print("\nMissing target values:")
print(df_final["Target"].isna().sum())

target_check = ((df_final["Future Return"] > 0).astype(int) == df_final["Target"])

print("\nTarget rule violations:")
print((~target_check).sum())

Target values:
Target
0    144
1    242
Name: count, dtype: int64

Unique target values:
[np.int64(0), np.int64(1)]

Missing target values:
0

Target rule violations:
0


## 13.2 Data Leakage Check

The prediction should use only information available at the prediction point.

The future-related columns are retained in the final dataset for auditability, but they must not be included as machine learning features.

In [ ]:
future_columns = ["One Year Date", "t1", "Price at t1", "Future Return", "Target"]

leakage_columns_in_features = [
    col for col in future_columns
    if col in candidate_features
]

print("Future columns included in features:")
print(leakage_columns_in_features)

Future columns included in features:
[]


## 13.3 Feature Set Sanity Check

The final machine learning input should contain only the selected predictive features.

Metadata, prediction-date information, future values, and the target variable are retained in the dataset for auditing but are excluded from the model inputs.

In [ ]:
print("Number of selected features:", len(candidate_features))

print("\nSelected features:")
for i, feature in enumerate(candidate_features, start=1):
    print(f"{i}. {feature}")

Number of selected features: 31

Selected features:
1. Historical Return
2. Historical Volatility
3. Average Trading Volume
4. Revenue
5. Gross Profit
6. Operating Income
7. Net Income
8. Revenue Growth
9. Gross Profit Margin
10. Operating Profit Margin
11. Net Profit Margin
12. Operating Income Growth
13. Net Income Growth
14. Operating Cash Flow
15. Capital Expenditure
16. Operating Cash Flow Growth
17. Free Cash Flow Growth
18. FCF Margin
19. Cash, Cash Equivalents & Short Term Investments
20. Total Current Assets
21. Total Assets
22. Total Current Liabilities
23. Total Liabilities
24. Total Equity
25. Short Term Debt
26. Long Term Debt
27. Current Ratio
28. Debt-to-Equity
29. Asset Growth
30. Equity Growth
31. Debt Growth


## 13.4 Preprocessing Sanity Check

Missing financial and market values were retained in the original dataset and handled after splitting the data into training and test sets.

We now verify that the imputed training and test datasets contain no remaining missing values.

In [ ]:
print("Missing values in training data:", X_train_imputed.isna().sum().sum())

print("Missing values in test data:", X_test_imputed.isna().sum().sum())

Missing values in training data: 0
Missing values in test data: 0


## 13.5 Train/Test Split Sanity Check

The dataset was divided into training and test sets before preprocessing.

The training set is used to fit the models and preprocessing steps, while the test set is kept separate for final evaluation.

We verify that the split contains all observations and that both datasets have the expected feature structure.

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTotal observations:", len(X_train) + len(X_test))
print("Expected observations:", len(df_final))

print("\nTraining features:", X_train.shape[1])
print("Test features:", X_test.shape[1])

X_train shape: (308, 31)
X_test shape: (78, 31)
y_train shape: (308,)
y_test shape: (78,)

Total observations: 386
Expected observations: 386

Training features: 31
Test features: 31


## 13.6 Scaling Sanity Check

Standardization was applied only to the Logistic Regression inputs.

The scaler was fitted using the training data and then used to transform both the training and test data. This prevents information from the test set from influencing the learned scaling parameters.

The Decision Tree was trained on imputed but unscaled features because tree-based models do not require standardization.

In [ ]:
print("Scaled training shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

print("\nTraining scaled mean (approximately):")
print(X_train_scaled.mean(axis=0)[:5])

print("\nTraining scaled standard deviation (approximately):")
print(X_train_scaled.std(axis=0)[:5])

Scaled training shape: (308, 31)
Scaled test shape: (78, 31)

Training scaled mean (approximately):
[-3.46043540e-17 -3.46043540e-16  5.76739234e-18  4.61391387e-17
  3.89298983e-17]

Training scaled standard deviation (approximately):
[1. 1. 1. 1. 1.]


## 13.7 Evaluation Consistency Check

All models should be evaluated using the same test set so that their performance can be compared fairly.

We verify that the test target contains the expected 78 observations and that the previously calculated predictions also contain 78 observations.

In [ ]:
print("Test observations:", len(y_test))

print("Baseline predictions:", len(baseline_predictions))
print("Logistic Regression predictions:", len(y_pred_logistic))
print("Decision Tree predictions:", len(y_pred_tree))

Test observations: 78
Baseline predictions: 78
Logistic Regression predictions: 78
Decision Tree predictions: 78


In [ ]:
print("Logistic Regression coefficients:", logistic_model.coef_.shape)

print("Decision Tree feature importances:", tree_model.feature_importances_.shape)

print("Expected number of features:", len(candidate_features))

Logistic Regression coefficients: (1, 31)
Decision Tree feature importances: (31,)
Expected number of features: 31


## 13.8 Prediction Output Sanity Check

Because this is a binary classification problem, the final predictions should contain only the two possible target classes: 0 and 1.

We verify the unique prediction values produced by the Logistic Regression and Decision Tree models.

In [ ]:
print("Logistic Regression prediction values:")
print(sorted(set(y_pred_logistic)))

print("\nDecision Tree prediction values:")
print(sorted(set(y_pred_tree)))

Logistic Regression prediction values:
[np.int64(0), np.int64(1)]

Decision Tree prediction values:
[np.int64(0), np.int64(1)]


## 13.9 Prediction Distribution Check

We examine the distribution of predicted classes from each model.

This helps verify that the models are producing both classes and allows us to compare their prediction behaviour with the actual test-set class distribution.

In [ ]:
print("Actual test-set distribution:")
print(y_test.value_counts().sort_index())

print("\nLogistic Regression prediction distribution:")
print(pd.Series(y_pred_logistic).value_counts().sort_index())

print("\nDecision Tree prediction distribution:")
print(pd.Series(y_pred_tree).value_counts().sort_index())

Actual test-set distribution:
Target
0    29
1    49
Name: count, dtype: int64

Logistic Regression prediction distribution:
0    15
1    63
Name: count, dtype: int64

Decision Tree prediction distribution:
0     9
1    69
Name: count, dtype: int64


## 13.10 Reproducibility Check

A reproducible machine learning workflow should produce the same results when the same data, preprocessing, model settings, and random seed are used.

We verify that retraining the Decision Tree with the same configuration produces the same predictions.

In [ ]:
tree_model_check = DecisionTreeClassifier(max_depth=4, random_state=42)

tree_model_check.fit(X_train_imputed, y_train)

y_pred_tree_check = tree_model_check.predict(X_test_imputed)

print("Predictions identical:", np.array_equal(y_pred_tree, y_pred_tree_check))

Predictions identical: True


## 13.11 Test-Set Alignment Check

Both models should be evaluated on exactly the same test observations.

We verify that the prediction arrays from Logistic Regression and the Decision Tree have the same length as the test target array.

In [ ]:
print("Test target length:", len(y_test))
print("Logistic Regression predictions:", len(y_pred_logistic))
print("Decision Tree predictions:", len(y_pred_tree))

print("\nAll lengths match:", len(y_test) == len(y_pred_logistic) == len(y_pred_tree))

Test target length: 78
Logistic Regression predictions: 78
Decision Tree predictions: 78

All lengths match: True


## 13.12 Prediction–Target Alignment Check

We verify that the predictions from each model contain one prediction for every test observation and that their ordering matches the test target array.

In [ ]:
print("Logistic Regression aligned:", len(y_pred_logistic) == len(y_test))

print("Decision Tree aligned:", len(y_pred_tree) == len(y_test))

Logistic Regression aligned: True
Decision Tree aligned: True


## 13.13 Metric Recalculation Check

We recalculate accuracy, precision, recall, and F1-score directly from the stored predictions.

This confirms that the evaluation results are consistent with the predictions produced by each model.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Logistic Regression:")
print("Accuracy:", accuracy_score(y_test, y_pred_logistic))
print("Precision:", precision_score(y_test, y_pred_logistic))
print("Recall:", recall_score(y_test, y_pred_logistic))
print("F1-score:", f1_score(y_test, y_pred_logistic))

print("\nDecision Tree:")
print("Accuracy:", accuracy_score(y_test, y_pred_tree))
print("Precision:", precision_score(y_test, y_pred_tree))
print("Recall:", recall_score(y_test, y_pred_tree))
print("F1-score:", f1_score(y_test, y_pred_tree))

Logistic Regression:
Accuracy: 0.5641025641025641
Precision: 0.6190476190476191
Recall: 0.7959183673469388
F1-score: 0.6964285714285714

Decision Tree:
Accuracy: 0.6410256410256411
Precision: 0.6521739130434783
Recall: 0.9183673469387755
F1-score: 0.7627118644067796


## 13.14 Confusion Matrix Consistency Check

We recalculate the confusion matrices directly from the test targets and model predictions.

This verifies that the reported true negatives, false positives, false negatives, and true positives are consistent with the stored predictions.

In [ ]:
from sklearn.metrics import confusion_matrix

cm_logistic_check = confusion_matrix(y_test, y_pred_logistic)
cm_tree_check = confusion_matrix(y_test, y_pred_tree)

print("Logistic Regression confusion matrix:")
print(cm_logistic_check)

print("\nDecision Tree confusion matrix:")
print(cm_tree_check)

Logistic Regression confusion matrix:
[[ 5 24]
 [10 39]]

Decision Tree confusion matrix:
[[ 5 24]
 [ 4 45]]


## 13.15 Majority Baseline Consistency Check

The majority-class baseline predicts the most common class in the training data for every test observation.

We verify that all baseline predictions are class 1 and that the number of predictions matches the test set.

In [ ]:
print("Number of baseline predictions:", len(baseline_predictions))

print("Unique baseline prediction values:", sorted(set(baseline_predictions)))

print("All baseline predictions are class 1:", np.all(baseline_predictions == 1))

Number of baseline predictions: 78
Unique baseline prediction values: [np.int64(1)]
All baseline predictions are class 1: True


## 13.16 Majority Baseline Metric Check

We recalculate the accuracy, precision, recall, and F1-score of the majority-class baseline.

This verifies that the baseline metrics are consistent with its predictions.

In [ ]:
print("Majority Baseline:")
print("Accuracy:", accuracy_score(y_test, baseline_predictions))
print("Precision:", precision_score(y_test, baseline_predictions))
print("Recall:", recall_score(y_test, baseline_predictions))
print("F1-score:", f1_score(y_test, baseline_predictions))

Majority Baseline:
Accuracy: 0.6282051282051282
Precision: 0.6282051282051282
Recall: 1.0
F1-score: 0.7716535433070866


## 13.17 Majority Baseline Confusion Matrix Check

We recalculate the confusion matrix directly from the test targets and the majority baseline predictions.

Because the baseline predicts class 1 for every observation, we expect no true negatives or false negatives.

In [ ]:
baseline_cm_check = confusion_matrix(y_test, baseline_predictions)

print("Majority Baseline confusion matrix:")
print(baseline_cm_check)

Majority Baseline confusion matrix:
[[ 0 29]
 [ 0 49]]


## 13.18 Class Distribution Consistency Check

The dataset contains two target classes: 0 and 1.

Because the train/test split used stratification, the proportion of each class should remain similar in the training and test sets.

We compare the class counts and proportions to verify this.

In [ ]:
print("Training target distribution:")
print(y_train.value_counts().sort_index())
print(y_train.value_counts(normalize=True).sort_index())

print("\nTest target distribution:")
print(y_test.value_counts().sort_index())
print(y_test.value_counts(normalize=True).sort_index())

Training target distribution:
Target
0    115
1    193
Name: count, dtype: int64
Target
0    0.373377
1    0.626623
Name: proportion, dtype: float64

Test target distribution:
Target
0    29
1    49
Name: count, dtype: int64
Target
0    0.371795
1    0.628205
Name: proportion, dtype: float64


## 13.19 End-to-End Prediction Sanity Check

We generate fresh predictions from the currently trained models using the current test inputs.

The new predictions should be identical to the predictions stored earlier in the notebook.

In [ ]:
logistic_predictions_check = logistic_model.predict(X_test_scaled)
tree_predictions_check = tree_model.predict(X_test_imputed)

print("Logistic Regression predictions identical:", np.array_equal(y_pred_logistic, logistic_predictions_check))

print("Decision Tree predictions identical:", np.array_equal(y_pred_tree, tree_predictions_check))

Logistic Regression predictions identical: True
Decision Tree predictions identical: True


# 14 Classical Machine Learning Conclusion

Three approaches were evaluated for the binary classification task: a majority-class baseline, Logistic Regression, and a Decision Tree.

The majority-class baseline achieved an accuracy of 62.82% by predicting the positive class for every test observation. Although it achieved 100% recall, it did not identify any non-positive-return companies and therefore provides a simple reference point rather than a practically useful model.

Logistic Regression achieved an accuracy of 56.41%, precision of 61.90%, recall of 79.59%, and an F1-score of 69.64%. The model identified most of the positive-return companies but produced a relatively high number of false positive predictions.

The Decision Tree achieved the strongest overall results among the two trained machine learning models, with an accuracy of 64.10%, precision of 65.22%, recall of 91.84%, and an F1-score of 76.27%. It also performed slightly better than the majority-class baseline in accuracy and precision.

The Decision Tree feature-importance analysis showed that Total Equity, Historical Return, Cash, Cash Equivalents & Short Term Investments, Debt-to-Equity, and Capital Expenditure were among the features used most by the trained tree. These importance scores describe the model's behaviour and should not be interpreted as evidence of causal relationships.

Overall, the Decision Tree is the most promising baseline model among the approaches tested. However, the results are based on a single train-test split and a relatively small dataset, so they should be considered initial baseline results rather than evidence of general performance.

The final dataset and preprocessing pipeline were also checked for duplicate observations, target errors, future-data leakage, missing values after imputation, and inconsistent feature sets. All sanity checks passed.


In [1]:
import os

ARTIFACT_DIR = (
    "/content/drive/MyDrive/"
    "data_handling_and_machine_learning/"
    "datasets/classical_ml_classification_dataset/"
)

files = [
    "InvestIQ_decision_tree_model.pkl",
    "InvestIQ_classical_ml_features.json",
    "InvestIQ_classical_ml_imputer.pkl",
]

for file in files:
    path = os.path.join(ARTIFACT_DIR, file)
    print(f"{file}: {'FOUND' if os.path.exists(path) else 'MISSING'}")

InvestIQ_decision_tree_model.pkl: FOUND
InvestIQ_classical_ml_features.json: FOUND
InvestIQ_classical_ml_imputer.pkl: FOUND


In [2]:
import json
import joblib

model = joblib.load(
    os.path.join(ARTIFACT_DIR, "InvestIQ_decision_tree_model.pkl")
)

imputer = joblib.load(
    os.path.join(ARTIFACT_DIR, "InvestIQ_classical_ml_imputer.pkl")
)

with open(
    os.path.join(ARTIFACT_DIR, "InvestIQ_classical_ml_features.json"),
    "r",
    encoding="utf-8"
) as f:
    features = json.load(f)

print("Model:", type(model).__name__)
print("Imputer:", type(imputer).__name__)
print("Number of features:", len(features))
print("First 5 features:", features[:5])

Model: DecisionTreeClassifier
Imputer: SimpleImputer
Number of features: 31
First 5 features: ['Historical Return', 'Historical Volatility', 'Average Trading Volume', 'Revenue', 'Gross Profit']


In [6]:
top_5, companies_scanned = investment_score.get_top_5(
    sector="All",
    risk="Moderate",
    goal="Balanced Return and Risk"
)

print("Companies scanned:", companies_scanned)
print()
print("Top 5 companies:")

for index in top_5.index:
    company_row = top_5.loc[[index]]

    input_data = company_row[features].copy()
    input_imputed = imputer.transform(input_data)

    input_imputed = pd.DataFrame(
        input_imputed,
        columns=features,
        index=input_data.index
    )

    probability = model.predict_proba(input_imputed)[0][1]

    print(
        f"{company_row['Ticker'].iloc[0]}: "
        f"{probability:.2%}"
    )

NameError: name 'investment_score' is not defined